In [1]:
!pip install transformers -U

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 108.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 89.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 1.0.0rc2
    Uninstalling huggingface-hub-1.0.0rc2:
      Successfully uninstalled huggingface-hub-1.0.0rc2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.2
    Uninstalling tokenizers-0.21.2:
      Successfully uninstalled tokenizers-0.21.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.53.3
    Uninstalling transformers-4.53.3:
      Successfully uninstalled transformers-4.53.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source

In [ ]:
import cv2
import torch
from PIL import Image
from collections import Counter
import transformers
from transformers import LlavaNextForConditionalGeneration, LlavaNextProcessor
import os
from tqdm import tqdm
import time
import csv
import warnings
import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()

2025-10-27 15:52:44.092667: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761580364.286613      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761580364.335558      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [ ]:
MODEL_ID = "llava-hf/llava-v1.6-mistral-7b-hf"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TORCH_DTYPE = torch.float16

print(f"Utilizzo del dispositivo: {DEVICE}")
print(f"Caricamento del modello: {MODEL_ID}")

# --- CARICAMENTO DEL MODELLO E DEL PROCESSORE ---
processor = LlavaNextProcessor.from_pretrained(MODEL_ID)
model = LlavaNextForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=TORCH_DTYPE,
    low_cpu_mem_usage=True,
    device_map="auto"
)

print("✅ Modello e processore LLaVA 1.6 caricati con successo.")

Utilizzo del dispositivo: cuda
Caricamento del modello: llava-hf/llava-v1.6-mistral-7b-hf


preprocessor_config.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/176 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

✅ Modello e processore LLaVA 1.6 caricati con successo.


In [4]:
VALID_EMOTIONS = ['joy', 'anger', 'fear', 'disgust', 'surprise', 'sadness', 'boredom', 'neutral', 'contentment']

In [ ]:
def analyze_holistic_emotion(image_pil):
    """
    Analizza l'emozione dominante dell'intera immagine utilizzando il chat template di LLaVA 1.6.
    Restituisce l'emozione, la risposta raw e il tempo di inferenza.
    """
    
    prompt_instructions = (
        "Identify the dominant group emotion in this image following these specific rules: "
        "1. Prioritize active emotions. For happiness, choose 'joy' (for smiles, laughter) over 'contentment' (for calm satisfaction). If in doubt between them, you MUST choose 'joy'. "
        "2. Distinguish 'sadness' (emotional pain, tears, downturned mouth, tense face) from 'boredom' (lack of stimulation, blank stare). If any sign of pain or distress is present, choose 'sadness'. "
        "3. Distinguish 'anger' (hostility, furrowed brows, tense jaw) from 'disgust' (revulsion, wrinkled nose, pulled-back lips). "
        "4. Use 'fear' only for reactions to a clear threat (defensive posture, shrinking back, wide eyes with tension). "
        "5. Use 'surprise' ONLY if the expression is *brief and shock-like*: wide eyes + open mouth + no signs of sustained pain, fear, anger, or disgust. "
        "   - If the same cues could indicate fear, sadness, or disgust, DO NOT choose 'surprise'. "
        "6. 'Neutral' is a LAST resort, only if there are no visible emotional cues at all. "
        f"Choose ONLY ONE or TWO emotion from the allowed list: {VALID_EMOTIONS}. "
        "Your answer MUST be in the format: 'emotions: [ em1, em2]'"
    )
    
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt_instructions},
                {"type": "image"},
            ],
        },
    ]
    prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)

    inputs = processor(images=image_pil, text=prompt, return_tensors="pt").to(DEVICE)
    
    inference_start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            do_sample=True,
            temperature=0.2,
            top_p=0.95,
            max_new_tokens=20
        )
    inference_end_time = time.time()
    frame_inference_time = inference_end_time - inference_start_time
    

    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]
    raw_response = processor.decode(generated_tokens, skip_special_tokens=True)
    emotion_text = raw_response.lower()
    
    found_emotions = [kw for kw in VALID_EMOTIONS if kw in emotion_text]
    
    if found_emotions:
        parsed_emotion = ", ".join(found_emotions)
    else:
        parsed_emotion = "nan"

    return parsed_emotion, raw_response, frame_inference_time 

In [ ]:
def draw_holistic_annotation(frame, holistic_emotion):
    """Disegna l'annotazione dell'emozione olistica sul frame."""
    text_holistic = f"Group Emotion: {holistic_emotion.upper()}"    
    cv2.rectangle(frame, (5, 5), (400, 40), (0, 0, 0), -1)    
    cv2.putText(frame, text_holistic, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    return frame

def write_summary_csv(summary_data, csv_path):
    """Scrive il file CSV di riepilogo finale."""
    if not summary_data:
        print("Nessun dato di riepilogo da scrivere.")
        return

    fieldnames = ['video_clip', 'holistic_dominant_emotion', 'holistic_distribution']
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(summary_data)
    print(f"🏆 Riepilogo finale salvato con successo in: {csv_path}")

In [ ]:
def run_holistic_analysis(video_path, base_output_dir, frames_per_second=2):
    """
    Esegue l'analisi olistica su un singolo file video.
    Restituisce il riepilogo, il tempo di inferenza totale e il conteggio dei frame.
    """
    clip_name = os.path.splitext(os.path.basename(video_path))[0]
    video_output_dir = os.path.join(base_output_dir, clip_name)
    frames_dir = os.path.join(video_output_dir, "frames")
    os.makedirs(frames_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Errore: Impossibile aprire il video {video_path}")
        return None, 0, 0  

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    output_video_path = os.path.join(video_output_dir, f"{clip_name}_holistic_annotated.mp4")
    video_writer = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_skip = int(fps / frames_per_second) if frames_per_second > 0 and fps > 0 else 1

    analysis_data = []
    last_holistic_emotion = "N/A"    
    
    total_inference_time_for_clip = 0.0
    analyzed_frames_count = 0    

    with tqdm(total=frame_count, desc=f"Analisi di {clip_name}") as pbar:
        frame_number = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break

            if frame_number % frame_skip == 0:
                frame_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                                
                holistic_emotion, holistic_raw, frame_inference_time = analyze_holistic_emotion(frame_pil)
                total_inference_time_for_clip += frame_inference_time
                analyzed_frames_count += 1
                
                
                last_holistic_emotion = holistic_emotion

                
                frame_filename = f"frame_{frame_number:05d}.jpg"
                frame_path = os.path.join(frames_dir, frame_filename)
                cv2.imwrite(frame_path, frame)
                
                analysis_data.append({
                    "frame_number": frame_number,
                    "frame_image_path": frame_path,
                    "holistic_emotion": holistic_emotion,
                    "holistic_raw_response": holistic_raw,
                })
            
            annotated_frame = draw_holistic_annotation(frame.copy(), last_holistic_emotion)
            video_writer.write(annotated_frame)
            pbar.update(1)
            frame_number += 1

    cap.release()
    video_writer.release()
    print(f"\n✅ Video annotato salvato in: {output_video_path}")

    
    csv_path = os.path.join(video_output_dir, f"holistic_report_{clip_name}.csv")
    if analysis_data:
        fieldnames = ["frame_number", "frame_image_path", "holistic_emotion", "holistic_raw_response"]
        with open(csv_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(analysis_data)
        print(f"✅ Report frame per frame salvato in: {csv_path}")

    if not analysis_data:
        return None, 0, 0  

    
    holistic_emotions_clip = [row['holistic_emotion'] for row in analysis_data if row['holistic_emotion'] != 'nan']    
    all_emotions = [e.strip() for emotions in holistic_emotions_clip for e in emotions.split(',')]
    
    holistic_distribution = dict(Counter(all_emotions))
    holistic_dominant = Counter(all_emotions).most_common(1)[0][0] if all_emotions else "N/A"

    summary = {
        'video_clip': clip_name,
        'holistic_dominant_emotion': holistic_dominant,
        'holistic_distribution': str(holistic_distribution),
    }

    return summary, total_inference_time_for_clip, analyzed_frames_count 

In [ ]:
if __name__ == "__main__":    
    INPUT_DIR = "/kaggle/input/video-final-1/"  
    OUTPUT_DIR = "/kaggle/working/holistic_results/"

    # Imposta quanti frame al secondo analizzare
    FRAMES_TO_ANALYZE_PER_SECOND = 8

    if not os.path.isdir(INPUT_DIR):
        print(f"⚠️ ERRORE: La cartella di input '{INPUT_DIR}' non esiste. Controlla il percorso.")
    else:
        video_files = [f for f in os.listdir(INPUT_DIR) if f.endswith(('.mp4', '.avi', '.mov'))]
        if not video_files:
            print(f"⚠️ Attenzione: Nessun file video trovato in '{INPUT_DIR}'.")
        else:
            os.makedirs(OUTPUT_DIR, exist_ok=True)            
            
            all_clips_summary_data = []
            total_script_inference_time = 0.0
            total_script_analyzed_frames = 0
            script_start_time = time.time()
            
            for video_file in video_files:
                input_video_path = os.path.join(INPUT_DIR, video_file)
                print(f"--- Inizio analisi olistica per: {video_file} ---")                
                
                summary_result, clip_inference_time, clip_analyzed_frames = run_holistic_analysis(
                    video_path=input_video_path,
                    base_output_dir=OUTPUT_DIR,
                    frames_per_second=FRAMES_TO_ANALYZE_PER_SECOND
                )
                
                
                if summary_result:
                    all_clips_summary_data.append(summary_result)                    
                    total_script_inference_time += clip_inference_time
                    total_script_analyzed_frames += clip_analyzed_frames                    
            
            
            summary_csv_path = os.path.join(OUTPUT_DIR, "final_holistic_summary_report.csv")
            write_summary_csv(all_clips_summary_data, summary_csv_path)
            
            print("\n\n--- 📝 Riepilogo Dati Clip (all_clips_summary_data) ---")
            print(all_clips_summary_data)            
            
            script_end_time = time.time()
            total_execution_time = script_end_time - script_start_time
            
            avg_inference_time_per_frame = 0.0
            if total_script_analyzed_frames > 0:
                avg_inference_time_per_frame = total_script_inference_time / total_script_analyzed_frames
            
            stats_csv_path = os.path.join(OUTPUT_DIR, "final_timing_statistics.csv")
            try:
                stats_data = {
                    'total_execution_time_seconds': f"{total_execution_time:.2f}",
                    'total_inference_time_seconds': f"{total_script_inference_time:.2f}",
                    'total_analyzed_frames': total_script_analyzed_frames,
                    'avg_inference_time_seconds_per_frame': f"{avg_inference_time_per_frame:.4f}"
                }
                fieldnames_stats = [
                    'total_execution_time_seconds', 
                    'total_inference_time_seconds', 
                    'total_analyzed_frames', 
                    'avg_inference_time_seconds_per_frame'
                ]
                
                with open(stats_csv_path, 'w', newline='', encoding='utf-8') as f_stats:
                    writer_stats = csv.DictWriter(f_stats, fieldnames=fieldnames_stats)
                    writer_stats.writeheader()
                    writer_stats.writerow(stats_data)
                print(f"\n✅ Statistiche di timing salvate su: {stats_csv_path}")
            except Exception as e:
                print(f"⚠️ Errore durante il salvataggio del CSV delle statistiche: {e}")            
            
            print("\n\n--- 📊 Statistiche di Analisi (Stampa finale) ---")
            print(f"⏱️ Tempo totale di esecuzione (I/O + inferenza): {total_execution_time:.2f} secondi")
            print(f"⏱️ Tempo totale di *inferenza* (solo modello): {total_script_inference_time:.2f} secondi")
            print(f"🖼️ Totale frame analizzati: {total_script_analyzed_frames}")
            print(f"⚡ Tempo medio di inferenza per frame: {avg_inference_time_per_frame:.4f} secondi/frame")            

            print("\n\n✨ Analisi olistica completata per tutti i video.")

--- Inizio analisi olistica per: dia108_utt2.mp4 ---


Analisi di dia108_utt2: 100%|██████████| 69/69 [05:12<00:00,  4.52s/it]



✅ Video annotato salvato in: /kaggle/working/holistic_results/dia108_utt2/dia108_utt2_holistic_annotated.mp4
✅ Report frame per frame salvato in: /kaggle/working/holistic_results/dia108_utt2/holistic_report_dia108_utt2.csv
--- Inizio analisi olistica per: dia235_utt2.mp4 ---


Analisi di dia235_utt2: 100%|██████████| 163/163 [11:52<00:00,  4.37s/it]



✅ Video annotato salvato in: /kaggle/working/holistic_results/dia235_utt2/dia235_utt2_holistic_annotated.mp4
✅ Report frame per frame salvato in: /kaggle/working/holistic_results/dia235_utt2/holistic_report_dia235_utt2.csv
--- Inizio analisi olistica per: dia354_utt15.mp4 ---


...


✅ Video annotato salvato in: /kaggle/working/holistic_results/dia682_utt7/dia682_utt7_holistic_annotated.mp4
✅ Report frame per frame salvato in: /kaggle/working/holistic_results/dia682_utt7/holistic_report_dia682_utt7.csv
--- Inizio analisi olistica per: dia38_utt0.mp4 ---


Analisi di dia38_utt0: 100%|██████████| 83/83 [06:12<00:00,  4.49s/it]



✅ Video annotato salvato in: /kaggle/working/holistic_results/dia38_utt0/dia38_utt0_holistic_annotated.mp4
✅ Report frame per frame salvato in: /kaggle/working/holistic_results/dia38_utt0/holistic_report_dia38_utt0.csv
--- Inizio analisi olistica per: dia66_utt4.mp4 ---


Analisi di dia66_utt4: 100%|██████████| 141/141 [07:10<00:00,  3.06s/it]


✅ Video annotato salvato in: /kaggle/working/holistic_results/dia66_utt4/dia66_utt4_holistic_annotated.mp4
✅ Report frame per frame salvato in: /kaggle/working/holistic_results/dia66_utt4/holistic_report_dia66_utt4.csv
🏆 Riepilogo finale salvato con successo in: /kaggle/working/holistic_results/final_holistic_summary_report.csv


--- 📝 Riepilogo Dati Clip (all_clips_summary_data) ---
[{'video_clip': 'dia108_utt2', 'holistic_dominant_emotion': 'sadness', 'holistic_distribution': "{'sadness': 26, 'neutral': 17, 'joy': 4, 'disgust': 1, 'anger': 3, 'surprise': 9, 'fear': 4}"}, {'video_clip': 'dia235_utt2', 'holistic_dominant_emotion': 'contentment', 'holistic_distribution': "{'joy': 60, 'contentment': 66, 'sadness': 12, 'neutral': 17, 'boredom': 1}"}, {'video_clip': 'dia354_utt15', 'holistic_dominant_emotion': 'neutral', 'holistic_distribution': "{'disgust': 32, 'sadness': 42, 'boredom': 1, 'neutral': 69}"}, {'video_clip': 'dia110_utt9', 'holistic_dominant_emotion': 'contentment', 'holist

In [ ]:
source_dir = '/kaggle/working/holistic_results'
output_filename = '/kaggle/working/llava_holistic_results'

if os.path.exists(source_dir):
    shutil.make_archive(output_filename, 'zip', source_dir)
    print(f"✅ La cartella '{source_dir}' è stata compressa con successo in '{output_filename}.zip'")
else:
    print(f"⚠️ Attenzione: La cartella '{source_dir}' non è stata trovata.")

✅ La cartella '/kaggle/working/holistic_results' è stata compressa con successo in '/kaggle/working/llava_holistic_results.zip'
